## A better take including the superscripted valences of the ions

In [12]:
import pandas as pd

# Define the file names
geochem_files = [
    'LOOP2_geochem_all_.csv',
    'LOOP3_geochem_all_.csv',
    'LOOP4_geochem_all_.csv',
    'LOOP6_geochem_all_.csv',
    'DEMO_geochem_all_.csv'
]
extra_file = 'DEMO_extra.csv'
output_file = 'Master_Geochemistry.csv'

# -----------------------------
# 1) Process the DEMO_extra file
# -----------------------------
df_extra = pd.read_csv(extra_file)

# Join keys for wedging
join_keys = ['ID', 'Depth (m)']

# Columns that should NOT become GW_* and should instead fill existing columns
merge_into_existing = ['LOOPNr', 'DGUnr', 'method']

# Add 'GW_' prefix to all columns from DEMO_extra EXCEPT:
# - join keys (needed for merge)
# - LOOPNr/DGUnr/method (these should populate existing columns, not create GW_*)
df_extra = df_extra.rename(columns={
    col: f"GW_{col}"
    for col in df_extra.columns
    if col not in join_keys and col not in merge_into_existing
})

# -------------------------------------------
# 2) Wedge the extra data into the DEMO geochem
# -------------------------------------------
df_demo_all = pd.read_csv('DEMO_geochem_all_.csv')

# Merge, keeping DEMO_geochem columns as primary (suffix extra columns only when collision occurs)
df_demo_updated = pd.merge(
    df_demo_all,
    df_extra,
    on=join_keys,
    how='outer',
    suffixes=('', '_extra')
)

# Coalesce LOOPNr/DGUnr/method from extra into existing columns (only fill missing)
for col in merge_into_existing:
    extra_col = f"{col}_extra"
    if col in df_demo_updated.columns and extra_col in df_demo_updated.columns:
        df_demo_updated[col] = df_demo_updated[col].combine_first(df_demo_updated[extra_col])
        df_demo_updated = df_demo_updated.drop(columns=[extra_col])

# ---------------------------------
# 3) Stitch all _geochem_all_ files
# ---------------------------------
all_dataframes = []
for file_name in geochem_files:
    if file_name == 'DEMO_geochem_all_.csv':
        all_dataframes.append(df_demo_updated)
    else:
        all_dataframes.append(pd.read_csv(file_name))

master_df = pd.concat(all_dataframes, ignore_index=True, sort=False)

# ----------------------------------------------------
# (1) Convert LOOPNR/LOOPNr values DEMO1/DEMO2 -> DEMO
# ----------------------------------------------------
for loop_col in ['LOOPNR', 'LOOPNr']:
    if loop_col in master_df.columns:
        master_df[loop_col] = master_df[loop_col].replace({'DEMO1': 'DEMO', 'DEMO2': 'DEMO'})

# -------------------------------------------
# (2) Fix sulfate header: SO₄²⁻ [mg/L]
# -------------------------------------------
sulfate_rename = {
    'SO2-4 [mg/L]': 'SO₄²⁻ [mg/L]',
    'GW_SO2-4 [mg/L]': 'GW_SO₄²⁻ [mg/L]'
}
master_df = master_df.rename(columns=sulfate_rename)

# -------------------------------------------
# De-duplicate SO₄²⁻ [mg/L]: keep ONE column
# -------------------------------------------
target = 'SO₄²⁻ [mg/L]'

# Find all columns with this exact name (can be duplicates in pandas)
dup_idx = [i for i, c in enumerate(master_df.columns) if c == target]

if len(dup_idx) > 1:
    # Take first non-null across the duplicated columns, row-wise
    combined = master_df.iloc[:, dup_idx].bfill(axis=1).iloc[:, 0]

    # Drop all duplicates, then add back a single clean column
    master_df = master_df.drop(columns=[target])
    master_df[target] = combined

# Move SO₄²⁻ [mg/L] to Excel column K (0-based index 10)
if target in master_df.columns:
    cols = list(master_df.columns)
    cols.remove(target)
    insert_pos = min(10, len(cols))   # safe even if fewer than 11 cols
    cols.insert(insert_pos, target)
    master_df = master_df[cols]

# --- Existing superscript polishing for cations ---
chemical_map = {
    "Al": "Al³⁺",
    "Ba": "Ba²⁺",
    "Ca": "Ca²⁺",
    "Fe": "Fe²⁺",
    "K":  "K⁺",
    "Mg": "Mg²⁺",
    "Mn": "Mn²⁺",
    "Na": "Na⁺",
    "Ni": "Ni²⁺",
    "Sr": "Sr²⁺"
}

rename_dict = {}
for elem, superscripted in chemical_map.items():
    rename_dict[f"{elem} [mg/L]"] = f"{superscripted} [mg/L]"
    rename_dict[f"GW_{elem} [mg/L]"] = f"GW_{superscripted} [mg/L]"

master_df = master_df.rename(columns=rename_dict)
# -----------------------------------------------

# Final Polish: Ensure 'GW_' columns are at the far right
shared_cols = [c for c in master_df.columns if not c.startswith('GW_')]
gw_cols = [c for c in master_df.columns if c.startswith('GW_')]
master_df = master_df[shared_cols + gw_cols]

# Save (utf-8-sig helps Excel display subscripts/superscripts correctly)
master_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Successfully created {output_file} with {len(master_df)} total records.")

Successfully created Master_Geochemistry.csv with 711 total records.
